# Altair playground
A testing area for Altair plots

In [ ]:
import altair as alt
from altair.datasets import data
from harpy.report.theme import get_okabe
import pandas as pd
import altair as alt
import pandas as pd
import numpy as np


In [ ]:
#| label mycelllabel
inversions = pd.DataFrame({
    'chromosome': ['chr1', 'chr1', 'chr2', 'chr2', 'chr3', 'chr1', 'chr3'],
    'start': [1000000, 5000000, 2000000, 8000000, 3000000, 7000000, 6000000],
    'end': [2000000, 6500000, 3500000, 9500000, 5000000, 8500000, 7500000],
    'variant': ['INV','INV','INV','INV','INV','INV','INV']

})

# Sample data: deletions
deletions = pd.DataFrame({
    'chromosome': ['chr1', 'chr2', 'chr2', 'chr3', 'chr3', 'chr1', 'chr2'],
    'start': [4000, 1000000, 6000000, 1000000, 6000000, 8000000, 10000000],
    'end': [6000, 1800000, 7000000, 2000000, 7000000, 9000000, 11000000],
    'variant': ['DEL','DEL','DEL','DEL','DEL','DEL','DEL'],
})


sv = pd.concat([inversions,deletions])
sv.head()


In [ ]:
# Define chromosome lengths (example values - adjust to your genome)
chr_lengths = {
    'chr1': 10000000,
    'chr2': 12000000,
    'chr3': 8000000
}
chrs = pd.DataFrame(chr_lengths.items(), columns = ["chromosome", "length"])
chrs.head()


In [ ]:
var_df = sv.merge(chrs, "left", on = "chromosome")
var_df.head()


In [ ]:
def by_chromosome_plot(variants: pd.DataFrame, title:str = "WOW"):
    labels = variants['chromosome'].unique()
    input_dropdown = alt.binding_select(options=labels, name='Chromosome: ')
    selection = alt.selection_point("chrom_choice", fields=['chromosome'], value=labels[0], bind=input_dropdown)
    length_param = alt.param(expr='data("data_0")[0].length')
    highlight = alt.selection_point(name="highlight", on="pointerover", empty=False)

    stroke_color = (
        alt.when(highlight).then(alt.value("#7ae00d"))
        .otherwise(alt.value("transparent"))
    )

    return (
        alt.Chart(variants)
        .mark_bar(strokeWidth=1.5, cornerRadius=8)
        .encode(
            x=alt.X('start:Q')
                .scale(alt.Scale(domain=[0, length_param]))
                .axis(alt.Axis(title='Position (Mb)', labelExpr='datum.value / 1000000')),
            x2='end:Q',
            y=alt.Y('variant:N', title = "Variant Type"),
            color=alt.Color('variant:N'),#.legend(None),
            tooltip=['variant:N', 'chromosome:N', 'start:Q', 'end:Q'],
            stroke=stroke_color
        )
        .transform_filter(selection)
        .add_params(selection, length_param, highlight)
        .properties(title= title)
    )

by_chromosome_plot(var_df)
